# Regnskapsenheter og restliste: etterprøvbar kontroll

Kjørt mot skrivebeskyttet produksjonsuttrekk 8. september 2026. Ingen database skrives fra denne notatboken. Sett `FOOD_FOLLOWUP_EVIDENCE` til mappen med uttrekkene dersom den flyttes.

Uttrekk: `scripts/verify-project-followup.mjs` bruker `BEGIN ISOLATION LEVEL REPEATABLE READ READ ONLY`. Migrasjonsprøven: `scripts/verify-financial-storage-migration.ts` oppretter midlertidige tabeller og ruller alltid tilbake. Dette er teknisk lagringsavstemming; kildeinnhold og menneskelig godkjenning behandles separat.


In [1]:
from pathlib import Path
import json, os, hashlib
from collections import Counter
base = Path(os.environ.get("FOOD_FOLLOWUP_EVIDENCE", '/Users/gabrielfreeman/.codex/visualizations/2026/09/08/01a08185-e2dc-73e2-a8d8-7d54a2bbe507/followup-2026-09-08'))
inventory_bytes = (base / "production-before.json").read_bytes()
inventory = json.loads(inventory_bytes)
replay = json.loads((base / "financial-migration-replay.json").read_text())
assert inventory["readOnly"] is True
print("Uttrekk:", inventory["checkedAt"])
print("SHA256:", hashlib.sha256(inventory_bytes).hexdigest())
print("Regnskapsrader:", len(inventory["financials"]))


Uttrekk: 2026-09-08T18:38:51.446Z
SHA256: 84fe825e0003a96c0c7945b35e3764e6a177529e81b6a1de8ef974b2b7e3e855
Regnskapsrader: 186


In [2]:
assert replay["sourceAndAuthorityHashBefore"] == replay["sourceAndAuthorityHashAfter"]
assert replay["rowCount"] == 186
assert replay["annotated"] == 176
print("Enhetsmerket:", replay["annotated"], "Uavklart:", len(replay["unknown"]))
for probe in replay["probes"]:
    print(probe)
print("Kildebeløp og autoritetsfelt er uendret i migrasjonsprøven.")


Enhetsmerket: 176 Uavklart: 10
{'orgNr': 'SE-556542-5353', 'year': 2025, 'scale': 1000000, 'amountCurrency': 'NOK', 'revenueNok': 94397690000}
{'orgNr': '815664582', 'year': 2024, 'scale': 1000000, 'amountCurrency': 'NOK', 'revenueNok': 579000000}
{'orgNr': '989278835', 'year': 2024, 'scale': 1000000, 'amountCurrency': 'NOK', 'revenueNok': 736000000}
Kildebeløp og autoritetsfelt er uendret i migrasjonsprøven.


In [3]:
print("Klassifiseringer:", Counter(r["status"] for r in inventory["populationInputs"]))
print("Kandidatkjøringer og vedtak:", inventory["candidateCounts"])
print("Åpne strømmer:", Counter(r["cellId"] for r in inventory["flows"] if r["quantity"] is None))
print("Leveranser uten kjøper:", sum(r["total"] - r["withBuyer"] for r in inventory["deliveries"]))
actors = inventory["actorFollowups"]
print("Aktøroppfølging:", len(actors))
for field in ["owner", "nextStep", "lastVerifiedAt"]:
    print("Mangler", field, sum(not r[field] for r in actors))
print("Semantisk søk:", inventory["semantic"])


Klassifiseringer: Counter({'approved_internal': 1355, 'review_required': 399, 'ai_draft': 15, 'blocked': 1})
Kandidatkjøringer og vedtak: {'CandidateContentUnit': 0, 'CandidateAnalysisRun': 0, 'CandidateAnalysisArtifact': 0, 'CandidateAssertion': 0, 'CandidateHumanReviewDecision': 0, 'CandidatePromotionDecision': 0}
Åpne strømmer: Counter({'food-waste-digestate': 20, 'seafood-residue-flow': 20})
Leveranser uten kjøper: 60310
Aktøroppfølging: 68
Mangler owner 23
Mangler nextStep 28
Mangler lastVerifiedAt 68
Semantisk søk: {'hasOpenAiKey': False, 'total': 1615, 'embedded': 0}
